# GEM-Bench Evaluation Analysis

Qualitative and Quantitative evaluation of ad-injected response quality across different **Base LLM** and **Judge LLM** combinations.

In [13]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

OUTPUT_BASE = '/home/gpuuser7/gpuuser7_a/prateek/GEM-Bench/GemBench/benchmarking/output/evaluate_results/output'

# Experiment configurations
EXPERIMENTS = [
    {
        'base_llm': 'gemma-3-27b-it',
        'judge': 'llama-3.3-70b',
        'dir': '20260926_210917_judge_llama-3.3-70b-instruct-fp8_base_gemma-3-27b-it',
    }
]

QUALITATIVE_METRICS = ['accuracy', 'naturalness', 'personality', 'trust', 'notice_products', 'click_products'] #, 'identifiability']
METRIC_DISPLAY = {
    'accuracy': 'Accuracy',
    'naturalness': 'Naturalness',
    'personality': 'Personality',
    'trust': 'Trust',
    'notice_products': 'Notice',
    'click_products': 'Click'
    # 'identifiability': 'Identifiability',
}

QUANTITATIVE_METRICS = ['local measure', 'global_coherence', 'global measure', 'ad_content_alignment', 'has_ad', 'product_selection_accuracy']
QUANT_METRIC_DISPLAY = {
    'local measure': 'RF',
    'global_coherence': 'RC',
    'global measure': 'AF',
    'ad_content_alignment': 'AC',
    'has_ad': 'IR',
    'product_selection_accuracy': 'CTR',
}


In [14]:
def load_results(xlsx_path):
    """Parse the evaluation_result_average.xlsx format."""
    df = pd.read_excel(xlsx_path, header=None)
    
    # Build column names from rows 1 (top-level) and 2 (metric names)
    row1 = df.iloc[1].tolist()
    row2 = df.iloc[2].tolist()
    cols = []
    for r1, r2 in zip(row1, row2):
        if pd.notna(r1) and str(r1) != 'nan':
            cols.append(str(r1))
        elif pd.notna(r2) and str(r2) != 'nan':
            cols.append(str(r2))
        else:
            cols.append('unknown')
    
    data = df.iloc[3:].reset_index(drop=True)
    data.columns = cols
    data['data_set'] = data['data_set'].ffill()
    
    # Convert metric columns to numeric
    for m in QUALITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    for m in QUANTITATIVE_METRICS:
        if m in data.columns:
            data[m] = pd.to_numeric(data[m], errors='coerce')
    
    return data

# Load all experiments
all_results = []
for exp in EXPERIMENTS:
    path = f"{OUTPUT_BASE}/{exp['dir']}/evaluation_result_average.xlsx"
    df = load_results(path)
    df['base_llm'] = exp['base_llm']
    df['judge'] = exp['judge']
    all_results.append(df)

combined = pd.concat(all_results, ignore_index=True)
print(f"Loaded {len(combined)} rows from {len(EXPERIMENTS)} experiments")
print(f"Datasets: {combined['data_set'].unique().tolist()}")
print(f"Solutions: {combined['solution'].unique().tolist()}")
print(f"Base LLMs: {combined['base_llm'].unique().tolist()}")
print(f"Judges: {combined['judge'].unique().tolist()}")


Loaded 15 rows from 1 experiments
Datasets: ['CA_Prod', 'LM-Market', 'MT-Human']
Solutions: ['Ad-Chat', 'GI-R', 'GI-P', 'GIR-R', 'GIR-P']
Base LLMs: ['gemma-3-27b-it']
Judges: ['llama-3.3-70b']


In [15]:
def build_paper_table(combined_df, dataset_filter=None, solution_filter=None):
    """Build a table in the paper format: Dataset | Base LLM | Judge LLM | Solution | Metrics."""
    
    df = combined_df.copy()
    if dataset_filter:
        df = df[df['data_set'].isin(dataset_filter)]
    if solution_filter:
        df = df[df['solution'].isin(solution_filter)]
    
    # Build the table rows
    rows = []
    for ds in df['data_set'].unique():
        ds_data = df[df['data_set'] == ds]
        for base in ds_data['base_llm'].unique():
            base_data = ds_data[ds_data['base_llm'] == base]
            for judge in base_data['judge'].unique():
                judge_data = base_data[base_data['judge'] == judge]
                for sol in judge_data['solution'].unique():
                    sol_data = judge_data[judge_data['solution'] == sol]
                    row = {
                        'Dataset': ds,
                        'Base LLM': base,
                        'Judge LLM': judge,
                        'Solution': sol,
                    }
                    vals_for_overall = []
                    for m in QUALITATIVE_METRICS:
                        val = sol_data[m].values[0]
                        if pd.notna(val) and str(val).lower() != 'nan':
                            row[METRIC_DISPLAY[m]] = round(float(val), 2)
                            vals_for_overall.append(float(val))
                        else:
                            row[METRIC_DISPLAY[m]] = '—'
                    
                    if len(vals_for_overall) > 0:
                        row['Overall'] = round(sum(vals_for_overall) / len(vals_for_overall), 2)
                    else:
                        row['Overall'] = '—'
                    rows.append(row)
    
    table = pd.DataFrame(rows)
    return table

# Build full table with Ad-Chat and GIR-R (matching paper format)


In [16]:
def style_table(df, title=''):
    """Style a DataFrame to bold the best value per (Dataset, Solution) group for each metric."""
    metric_cols = list(METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


In [17]:
def style_quant_table(df, title=''):
    metric_cols = list(QUANT_METRIC_DISPLAY.values()) + ['Overall']
    
    def bold_max_by_dataset(data):
        styles = pd.DataFrame('', index=data.index, columns=data.columns)
        for ds in data['Dataset'].unique():
            ds_idx = data[data['Dataset'] == ds].index
            for col in metric_cols:
                if col in data.columns:
                    numeric_s = pd.to_numeric(data.loc[ds_idx, col], errors='coerce')
                    if numeric_s.notna().any():
                        max_val = numeric_s.max()
                        is_max = numeric_s == max_val
                        styles.loc[ds_idx[is_max], col] = 'font-weight: bold'
        return styles
    
    styled = (df.style
              .apply(bold_max_by_dataset, axis=None)
              .set_caption(title)
              .format(precision=2)
              .set_table_styles([
                  {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('padding', '8px')]},
                  {'selector': 'td', 'props': [('padding', '6px 12px')]},
              ])
              .hide(axis='index'))
    return styled


## Results by Base LLM and Judge LLM

Displays the Qualitative and Quantitative metric tables for each unique combination of Base LLM and Judge LLM.

In [18]:
combinations = combined[['base_llm', 'judge']].drop_duplicates().values
for base, judge in combinations:
    subset = combined[(combined['base_llm'] == base) & (combined['judge'] == judge)]
    if len(subset) > 0:
        print(f"\n{'='*60}\nBase: {base} | Judge: {judge}\n{'='*60}")
        
        # Qualitative
        qual_table = build_paper_table(subset)
        display(style_table(qual_table, title=f'Qualitative Metrics (Base: {base}, Judge: {judge})'))
        
        # Quantitative
        quant_table = build_quant_paper_table(subset)
        display(style_quant_table(quant_table, title=f'Quantitative Metrics (Base: {base}, Judge: {judge})'))



Base: gemma-3-27b-it | Judge: llama-3.3-70b


Dataset,Base LLM,Judge LLM,Solution,Accuracy,Naturalness,Personality,Trust,Notice,Click,Overall
CA_Prod,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,61.17,39.58,51.67,32.25,66.17,84.42,55.88
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-R,71.11,19.51,42.87,28.11,30.16,81.15,45.48
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GI-P,71.61,20.08,43.80,28.68,30.17,81.07,45.90
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-R,76.68,32.79,55.16,33.93,47.54,80.66,54.46
CA_Prod,gemma-3-27b-it,llama-3.3-70b,GIR-P,77.31,33.31,56.03,34.13,47.52,80.91,54.87
LM-Market,gemma-3-27b-it,llama-3.3-70b,Ad-Chat,83.55,55.91,70.22,53.49,65.00,81.29,68.24
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-R,89.14,64.68,80.70,75.22,52.31,75.70,72.96
LM-Market,gemma-3-27b-it,llama-3.3-70b,GI-P,89.68,62.26,80.91,75.16,52.63,74.62,72.54
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-R,89.68,66.18,77.20,71.61,61.08,76.34,73.68
LM-Market,gemma-3-27b-it,llama-3.3-70b,GIR-P,89.57,66.72,76.72,71.29,59.30,75.43,73.17


NameError: name 'build_quant_paper_table' is not defined